# 🔥 Diffusion Models

In this notebook, we'll walk through the steps required to train your own diffusion model on the Oxford flowers dataset

The code is adapted from the excellent ['Denoising Diffusion Implicit Models' tutorial](https://keras.io/examples/generative/ddim/) created by András Béres available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2
# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-colorblind")

import copy
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, ConcatDataset, DataLoader
from PIL import Image
from safetensors.torch import load_file
from transformers import Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display, sample_batch

# The Hugging Face Trainer picks the best available device
# automatically, so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE = 64
DATASET_REPETITIONS = 5
LOAD_MODEL = False

NOISE_EMBEDDING_SIZE = 32
PLOT_DIFFUSION_STEPS = 20

# optimization
EMA = 0.999
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 50

## 1. Prepare the Data

In [ ]:
# Load the data
class ImageDataset(Dataset):
    """Every image found (recursively) under `root`, resized, without labels.
    The PyTorch counterpart of Keras' image_dataset_from_directory(labels=None)."""

    EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

    def __init__(self, root, image_size, transform=None):
        self.paths = sorted(
            p for p in Path(root).rglob("*") if p.suffix.lower() in self.EXTENSIONS
        )
        self.image_size = image_size
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        # (H, W, C) uint8 array -> (C, H, W) tensor: PyTorch images are channels-first
        img = torch.from_numpy(np.array(img)).permute(2, 0, 1)
        if self.transform is not None:
            img = self.transform(img)
        return img


train_data = ImageDataset(
    "../../../data/pytorch-challange-flower-dataset/dataset", image_size=IMAGE_SIZE
)
print(f"Found {len(train_data)} files.")

In [ ]:
# Preprocess the data
def preprocess(img):
    img = img.float() / 255.0
    return img


train_data.transform = preprocess  # applied to every image as it is loaded
train = ConcatDataset([train_data] * DATASET_REPETITIONS)  # like .repeat()
# No DataLoader needed: the Trainer batches and shuffles the dataset itself


# The Trainer passes each batch to the model as keyword arguments,
# so the collator turns a list of images into {"pixel_values": batch}
def data_collator(images):
    return {"pixel_values": torch.stack(images)}

In [ ]:
# Show some items of clothing from the training set
# (a DataLoader just to look at one batch; the Trainer makes its own for training)
train_sample = sample_batch(DataLoader(train, batch_size=BATCH_SIZE, shuffle=True))
display(train_sample)

### 1.1 Diffusion schedules <a name="diffusion_schedules"></a>

In [ ]:
def linear_diffusion_schedule(diffusion_times):
    min_rate = 0.0001
    max_rate = 0.02
    betas = min_rate + diffusion_times * (max_rate - min_rate)
    alphas = 1 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    signal_rates = torch.sqrt(alpha_bars)
    noise_rates = torch.sqrt(1 - alpha_bars)
    return noise_rates, signal_rates

In [ ]:
def cosine_diffusion_schedule(diffusion_times):
    signal_rates = torch.cos(diffusion_times * math.pi / 2)
    noise_rates = torch.sin(diffusion_times * math.pi / 2)
    return noise_rates, signal_rates

In [ ]:
def offset_cosine_diffusion_schedule(diffusion_times):
    min_signal_rate = 0.02
    max_signal_rate = 0.95
    start_angle = math.acos(max_signal_rate)
    end_angle = math.acos(min_signal_rate)

    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

    signal_rates = torch.cos(diffusion_angles)
    noise_rates = torch.sin(diffusion_angles)

    return noise_rates, signal_rates

In [ ]:
T = 1000
diffusion_times = torch.tensor([x / T for x in range(T)])
linear_noise_rates, linear_signal_rates = linear_diffusion_schedule(
    diffusion_times
)
cosine_noise_rates, cosine_signal_rates = cosine_diffusion_schedule(
    diffusion_times
)
(
    offset_cosine_noise_rates,
    offset_cosine_signal_rates,
) = offset_cosine_diffusion_schedule(diffusion_times)

In [ ]:
plt.plot(
    diffusion_times, linear_signal_rates**2, linewidth=1.5, label="linear"
)
plt.plot(
    diffusion_times, cosine_signal_rates**2, linewidth=1.5, label="cosine"
)
plt.plot(
    diffusion_times,
    offset_cosine_signal_rates**2,
    linewidth=1.5,
    label="offset_cosine",
)

plt.xlabel("t/T", fontsize=12)
plt.ylabel(r"$\bar{\alpha_t}$ (signal)", fontsize=12)
plt.legend()
plt.show()

In [ ]:
plt.plot(
    diffusion_times, linear_noise_rates**2, linewidth=1.5, label="linear"
)
plt.plot(
    diffusion_times, cosine_noise_rates**2, linewidth=1.5, label="cosine"
)
plt.plot(
    diffusion_times,
    offset_cosine_noise_rates**2,
    linewidth=1.5,
    label="offset_cosine",
)

plt.xlabel("t/T", fontsize=12)
plt.ylabel(r"$1-\bar{\alpha_t}$ (noise)", fontsize=12)
plt.legend()
plt.show()

## 2. Build the model <a name="build"></a>

In [ ]:
def sinusoidal_embedding(x):
    # x: noise variances of shape (batch, 1, 1, 1)
    frequencies = torch.exp(
        torch.linspace(
            math.log(1.0),
            math.log(1000.0),
            NOISE_EMBEDDING_SIZE // 2,
            device=x.device,
        )
    )
    # put the frequencies on the channel axis (dim 1 in PyTorch's NCHW layout)
    angular_speeds = 2.0 * math.pi * frequencies.view(1, -1, 1, 1)
    embeddings = torch.cat(
        [torch.sin(angular_speeds * x), torch.cos(angular_speeds * x)], dim=1
    )
    return embeddings  # (batch, NOISE_EMBEDDING_SIZE, 1, 1)

In [ ]:
embedding_list = []
for y in np.arange(0, 1, 0.01):
    x = torch.full((1, 1, 1, 1), y)
    embedding_list.append(sinusoidal_embedding(x)[0, :, 0, 0])
embedding_array = torch.stack(embedding_list).T.numpy()
fig, ax = plt.subplots()
ax.set_xticks(
    np.arange(0, 100, 10), labels=np.round(np.arange(0.0, 1.0, 0.1), 1)
)
ax.set_ylabel("embedding dimension", fontsize=8)
ax.set_xlabel("noise variance", fontsize=8)
plt.pcolor(embedding_array, cmap="coolwarm")
plt.colorbar(orientation="horizontal", label="embedding value")
ax.imshow(embedding_array, interpolation="nearest", origin="lower")
plt.show()

In [ ]:
# Unlike Keras, PyTorch layers need their input channel count up front,
# so every block takes `in_width` as well as its output `width`.
class ResidualBlock(nn.Module):
    def __init__(self, in_width, width):
        super().__init__()
        if in_width == width:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Conv2d(in_width, width, kernel_size=1)
        # affine=False is Keras' center=False, scale=False;
        # Keras momentum 0.99 == PyTorch momentum 0.01
        self.norm = nn.BatchNorm2d(in_width, affine=False, momentum=0.01, eps=1e-3)
        self.conv_1 = nn.Conv2d(in_width, width, kernel_size=3, padding="same")
        self.conv_2 = nn.Conv2d(width, width, kernel_size=3, padding="same")

    def forward(self, x):
        residual = self.residual(x)
        x = self.norm(x)
        x = F.silu(self.conv_1(x))  # silu == swish
        x = self.conv_2(x)
        return x + residual


class DownBlock(nn.Module):
    def __init__(self, in_width, width, block_depth):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ResidualBlock(in_width if i == 0 else width, width) for i in range(block_depth)]
        )
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x, skips):
        for block in self.blocks:
            x = block(x)
            skips.append(x)  # saved for the matching UpBlock
        return self.pool(x)


class UpBlock(nn.Module):
    def __init__(self, in_width, width, block_depth):
        super().__init__()
        # each skip connection has `width` channels and is concatenated to x
        self.blocks = nn.ModuleList(
            [
                ResidualBlock((in_width if i == 0 else width) + width, width)
                for i in range(block_depth)
            ]
        )
        self.up = nn.Upsample(scale_factor=2, mode="bilinear")

    def forward(self, x, skips):
        x = self.up(x)
        for block in self.blocks:
            x = torch.cat([x, skips.pop()], dim=1)  # concatenate on channels
            x = block(x)
        return x

In [ ]:
# Build the U-Net
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_in = nn.Conv2d(3, 32, kernel_size=1)
        self.down_1 = DownBlock(32 + NOISE_EMBEDDING_SIZE, 32, block_depth=2)
        self.down_2 = DownBlock(32, 64, block_depth=2)
        self.down_3 = DownBlock(64, 96, block_depth=2)
        self.mid_1 = ResidualBlock(96, 128)
        self.mid_2 = ResidualBlock(128, 128)
        self.up_1 = UpBlock(128, 96, block_depth=2)
        self.up_2 = UpBlock(96, 64, block_depth=2)
        self.up_3 = UpBlock(64, 32, block_depth=2)
        self.conv_out = nn.Conv2d(32, 3, kernel_size=1)
        nn.init.zeros_(self.conv_out.weight)  # kernel_initializer="zeros"
        nn.init.zeros_(self.conv_out.bias)

    def forward(self, noisy_images, noise_variances):
        x = self.conv_in(noisy_images)

        noise_embedding = sinusoidal_embedding(noise_variances)
        # (batch, 32, 1, 1) -> (batch, 32, IMAGE_SIZE, IMAGE_SIZE)
        noise_embedding = F.interpolate(
            noise_embedding, size=x.shape[-2:], mode="nearest"
        )

        x = torch.cat([x, noise_embedding], dim=1)

        skips = []

        x = self.down_1(x, skips)
        x = self.down_2(x, skips)
        x = self.down_3(x, skips)

        x = self.mid_1(x)
        x = self.mid_2(x)

        x = self.up_1(x, skips)
        x = self.up_2(x, skips)
        x = self.up_3(x, skips)

        return self.conv_out(x)


unet = UNet()
print(f"Total params: {sum(p.numel() for p in unet.parameters()):,}")

In [ ]:
class Normalizer(nn.Module):
    """PyTorch version of Keras layers.Normalization: rescales each colour
    channel to mean 0 and variance 1, using statistics computed from the data."""

    def __init__(self, num_channels=3):
        super().__init__()
        # Buffers are saved in the state_dict and moved by .to(device),
        # but they are not trainable parameters
        self.register_buffer("mean", torch.zeros(1, num_channels, 1, 1))
        self.register_buffer("variance", torch.ones(1, num_channels, 1, 1))

    @torch.no_grad()
    def adapt(self, data):
        # One pass over the data, accumulating per-channel sums (in float64)
        count, total, total_sq = 0, 0.0, 0.0
        for images in data:
            images = images.double()
            count += images.numel() // images.shape[1]  # pixels per channel
            total += images.sum(dim=(0, 2, 3))
            total_sq += (images**2).sum(dim=(0, 2, 3))
        mean = total / count
        variance = total_sq / count - mean**2  # Var[x] = E[x^2] - E[x]^2
        self.mean.copy_(mean.view(1, -1, 1, 1))
        self.variance.copy_(variance.view(1, -1, 1, 1))

    def forward(self, images):
        return (images - self.mean) / self.variance.sqrt().clamp(min=1e-7)


class DiffusionModel(nn.Module):
    def __init__(self, network):
        super().__init__()

        self.normalizer = Normalizer()
        self.network = network
        # The EMA network is a copy whose weights slowly track the trained
        # network; it is never trained directly, only used for sampling
        self.ema_network = copy.deepcopy(network)
        self.ema_network.requires_grad_(False)
        self.diffusion_schedule = offset_cosine_diffusion_schedule
        self.loss_fn = nn.L1Loss()  # mean absolute error

    def denormalize(self, images):
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return torch.clamp(images, 0.0, 1.0)

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        if training:
            network = self.network
        else:
            network = self.ema_network
        # train/eval mode switches BatchNorm between batch and running statistics
        network.train(training)
        pred_noises = network(noisy_images, noise_rates**2)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates

        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps
        current_images = initial_noise
        for step in range(diffusion_steps):
            diffusion_times = (
                torch.ones((num_images, 1, 1, 1), device=initial_noise.device)
                - step * step_size
            )
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images = self.denoise(
                current_images, noise_rates, signal_rates, training=False
            )
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(
                next_diffusion_times
            )
            current_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )
        return pred_images

    @torch.no_grad()  # sampling only, no gradients needed
    def generate(self, num_images, diffusion_steps, initial_noise=None):
        device = self.normalizer.mean.device
        if initial_noise is None:
            initial_noise = torch.randn(
                (num_images, 3, IMAGE_SIZE, IMAGE_SIZE), device=device
            )
        initial_noise = torch.as_tensor(initial_noise, dtype=torch.float32, device=device)
        generated_images = self.reverse_diffusion(
            initial_noise, diffusion_steps
        )
        generated_images = self.denormalize(generated_images)
        return generated_images

    # The Trainer calls forward(**batch) and expects the loss in the output
    def forward(self, pixel_values):
        """Add random amounts of noise to the images, predict that noise
        and return the loss between the true and predicted noise."""
        images = self.normalizer(pixel_values)
        noises = torch.randn_like(images)

        diffusion_times = torch.rand((images.shape[0], 1, 1, 1), device=images.device)
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)

        noisy_images = signal_rates * images + noise_rates * noises

        # train the network to separate noisy images to their components
        # (self.training is True inside trainer.train(), False after model.eval())
        pred_noises, pred_images = self.denoise(
            noisy_images, noise_rates, signal_rates, training=self.training
        )
        noise_loss = self.loss_fn(pred_noises, noises)
        return {"loss": noise_loss}

    @torch.no_grad()
    def update_ema(self):
        # ema_weight = EMA * ema_weight + (1 - EMA) * weight, for the weights and
        # the BatchNorm running statistics (Keras' `weights` includes both)
        ema_state = self.ema_network.state_dict().values()
        state = self.network.state_dict().values()
        for ema_weight, weight in zip(ema_state, state):
            if ema_weight.dtype.is_floating_point:
                ema_weight.mul_(EMA).add_(weight, alpha=1 - EMA)
            else:  # e.g. BatchNorm's integer batch counter
                ema_weight.copy_(weight)

In [ ]:
ddm = DiffusionModel(unet)
# one pass over the (un-repeated) images is enough to get the statistics
ddm.normalizer.adapt(DataLoader(train_data, batch_size=BATCH_SIZE))
print(f"mean: {ddm.normalizer.mean.flatten()}, variance: {ddm.normalizer.variance.flatten()}")

In [ ]:
if LOAD_MODEL:
    # The Trainer saves each checkpoint as ./checkpoint/checkpoint-<step>/model.safetensors
    checkpoint_dir = get_last_checkpoint("./checkpoint")
    ddm.load_state_dict(load_file(os.path.join(checkpoint_dir, "model.safetensors")))

## 3.Train the model <a name="train"></a>

In [ ]:
# TrainingArguments holds the optimizer and training settings. The Trainer uses
# AdamW and only optimizes parameters with requires_grad=True, i.e. the main
# network (the EMA copy is frozen). Note: it skips weight decay on biases.
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"  # where report_to="tensorboard" writes
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # the Trainer clips gradients by default; Keras doesn't
    per_device_train_batch_size=BATCH_SIZE,
    dataloader_drop_last=True,  # like drop_remainder=True
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    save_strategy="epoch",  # a checkpoint every epoch
    save_only_model=True,  # weights only, like save_weights_only=True
    save_total_limit=1,  # keep just the latest checkpoint
    report_to="tensorboard",
    remove_unused_columns=False,  # our batches are already what the model needs
)

In [ ]:
# run training and plot generated images periodically

# The plain Trainer knows nothing about EMA weights, so a callback updates them
# after every optimizer step (on_step_end runs right after optimizer.step())
class EMAUpdater(TrainerCallback):
    def on_step_end(self, args, state, control, model=None, **kwargs):
        model.update_ema()


class ImageGenerator(TrainerCallback):
    def __init__(self, num_img):
        self.num_img = num_img

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch) - 1  # state.epoch counts completed epochs
        generated_images = model.generate(
            num_images=self.num_img,
            diffusion_steps=PLOT_DIFFUSION_STEPS,
        ).cpu()
        display(
            generated_images,
            save_to="./output/generated_img_%03d.png" % (epoch),
        )


image_generator_callback = ImageGenerator(num_img=10)

trainer = Trainer(
    model=ddm,
    args=training_args,
    train_dataset=train,
    data_collator=data_collator,
    callbacks=[EMAUpdater(), image_generator_callback],
)
trainer.train()

## 4. Inference <a name="inference"></a>

In [ ]:
# Generate some novel images of flowers
generated_images = ddm.generate(num_images=10, diffusion_steps=20).cpu()
display(generated_images)

In [ ]:
# View improvement over greater number of diffusion steps
for diffusion_steps in list(np.arange(1, 6, 1)) + [20] + [100]:
    torch.manual_seed(42)
    generated_images = ddm.generate(
        num_images=10,
        diffusion_steps=diffusion_steps,
    ).cpu()
    display(generated_images)

In [ ]:
# Interpolation between two points in the latent space
torch.manual_seed(100)


def spherical_interpolation(a, b, t):
    return np.sin(t * math.pi / 2) * a + np.cos(t * math.pi / 2) * b


for i in range(5):
    a = torch.randn((3, IMAGE_SIZE, IMAGE_SIZE))
    b = torch.randn((3, IMAGE_SIZE, IMAGE_SIZE))
    initial_noise = torch.stack(
        [spherical_interpolation(a, b, t) for t in np.arange(0, 1.1, 0.1)]
    )
    generated_images = ddm.generate(
        num_images=2, diffusion_steps=20, initial_noise=initial_noise
    ).cpu()
    display(generated_images, n=11)